In [2]:

import tensorflow as tf 
import numpy as np
import matplotlib.pyplot as plt 
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report,confusion_matrix
import os 

In [6]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20

train_dir ="/home/eng/Desktop/project_osam/archive/chest_xray/chest_xray/train"
val_dir = "/home/eng/Desktop/project_osam/archive/chest_xray/chest_xray/val"
test_dir = "/home/eng/Desktop/project_osam/archive/chest_xray/chest_xray/test"

In [7]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True

)

val_datagen = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_generator = val_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 5216 images belonging to 2 classes.
Found 276 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


In [8]:
classes = train_generator.classes
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(classes),
    y=classes
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)



Class Weights: {0: np.float64(1.9448173005219984), 1: np.float64(0.6730322580645162)}


In [9]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

model = tf.keras.Sequential([
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

model.summary()


E0000 00:00:1772053477.087284   55873 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1772053477.097189   55873 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 67s 7us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,259,265 (8.62 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [10]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=3
    )
]


In [11]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks
)


Epoch 1/20


2026-02-26 00:06:59.634326: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 51380224 exceeds 10% of free system memory.
2026-02-26 00:06:59.690980: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 51380224 exceeds 10% of free system memory.
2026-02-26 00:06:59.746849: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 154140672 exceeds 10% of free system memory.
2026-02-26 00:06:59.833024: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 156905472 exceeds 10% of free system memory.
2026-02-26 00:06:59.878975: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 38535168 exceeds 10% of free system memory.


163/163 ━━━━━━━━━━━━━━━━━━━━ 142s 849ms/step - accuracy: 0.5784 - auc: 0.5954 - loss: 0.7154 - val_accuracy: 0.8804 - val_auc: 0.9662 - val_loss: 0.4550 - learning_rate: 1.0000e-04
Epoch 2/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 121s 739ms/step - accuracy: 0.7701 - auc: 0.8461 - loss: 0.4960 - val_accuracy: 0.9529 - val_auc: 0.9912 - val_loss: 0.3260 - learning_rate: 1.0000e-04
Epoch 3/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 121s 739ms/step - accuracy: 0.8386 - auc: 0.9198 - loss: 0.3891 - val_accuracy: 0.9420 - val_auc: 0.9932 - val_loss: 0.2647 - learning_rate: 1.0000e-04
Epoch 4/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 121s 743ms/step - accuracy: 0.8564 - auc: 0.9350 - loss: 0.3452 - val_accuracy: 0.9529 - val_auc: 0.9941 - val_loss: 0.2252 - learning_rate: 1.0000e-04
Epoch 5/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 121s 740ms/step - accuracy: 0.8806 - auc: 0.9565 - loss: 0.2926 - val_accuracy: 0.9493 - val_auc: 0.9945 - val_loss: 0.2054 - learning_rate: 1.0000e-04
Epoch 6/20
163/163 ━━━━━━━━━━━━━━━━━━━━ 122s 745ms/

In [ ]:
base_model.trainable = True

for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 144s 851ms/step - accuracy: 0.9080 - auc: 0.9695 - loss: 0.2277 - val_accuracy: 0.9058 - val_auc: 0.9954 - val_loss: 0.2112 - learning_rate: 1.0000e-05
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 138s 848ms/step - accuracy: 0.9337 - auc: 0.9854 - loss: 0.1589 - val_accuracy: 0.9094 - val_auc: 0.9936 - val_loss: 0.2177 - learning_rate: 1.0000e-05
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 135s 824ms/step - accuracy: 0.9456 - auc: 0.9893 - loss: 0.1338 - val_accuracy: 0.8986 - val_auc: 0.9945 - val_loss: 0.2474 - learning_rate: 1.0000e-05
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 134s 822ms/step - accuracy: 0.9498 - auc: 0.9898 - loss: 0.1290 - val_accuracy: 0.9239 - val_auc: 0.9975 - val_loss: 0.2008 - learning_rate: 2.0000e-06
Epoch 5/10
 93/163 ━━━━━━━━━━━━━━━━━━━━ 55s 790ms/step - accuracy: 0.9518 - auc: 0.9928 - loss: 0.1116

In [ ]:
loss, acc, auc = model.evaluate(test_generator)
print(f"\nTest Accuracy: {acc:.4f}")
print(f"Test AUC: {auc:.4f}")

# Predictions
preds = model.predict(test_generator)
pred_labels = (preds > 0.5).astype(int)

print("\nClassification Report:")
print(classification_report(test_generator.classes, pred_labels))

print("\nConfusion Matrix:")
print(confusion_matrix(test_generator.classes, pred_labels))


In [ ]:
model.save("pneumonia_model.keras")
print("Model saved as pneumonia_model.h5")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("pneumonia_model.tflite", "wb") as f:
    f.write(tflite_model)

print("Model converted to pneumonia_model.tflite")